# Notebook 3 : Projections, bases d'ondelettes NMF (+QR), SVD, GAM & Co

In [2]:
%reload_ext autoreload
%autoreload 2
import sys, pathlib, time
ROOT = pathlib.Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

import numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, IntSlider, SelectionSlider, Dropdown
from tqdm.auto import tqdm
from functools import lru_cache

import online_dp as dp
from online_dp.config import Config
from online_dp.cache import compute_or_load
from online_dp import data, metrics, basis, gam, mechanisms, vfast, viz

cfg = Config(
    data_dir=str(ROOT.parent / "data" / "DataDiffusionDeepCourboGen") + "/",  
    cache_dir=str(ROOT / "cache"),
    n_panel=500, n_target=50, seed=0,            
)
dp.cache.CACHE = pathlib.Path(cfg.cache_dir)
cfg

Config(data_dir='/home/G70186/data/DataDiffusionDeepCourboGen/', cache_dir='/home/G70186/onlinedp_stage26/cache', N=1000, n_panel=500, n_target=50, seed=0, calendar_start='2022-10-02 20:00:00', slots_per_day=48, pmax=47, n_groups=100, fit_subsample=10000, nmf_max_iter=500, nmf_tol=0.0001, clip_quantile=0.95, delta_dp=1e-05)

In [3]:
D = compute_or_load(cfg.key('dataset'), lambda: data.build_dataset(cfg))
df_daily, df_agg, df_temp, df_label = D['df_daily'], D['df_agg'], D['df_temp'], D['df_label']
panel_tensor, panel_ids = D['panel_tensor'], D['panel_ids']
panel_profiles = D['panel_profiles']
target_users   = D['target_users']
dates          = D['dates']
print('panel', panel_tensor.shape, '| agrege cible', df_agg.shape, '| cible', len(target_users))

[cache] 'dataset__N1000_np500_nt50_s0' rechargé (joblib).
panel (500, 362, 48) | agrege cible (362, 48) | cible 50


## Bases d'ondelettes matchées à la taille cible (n_target = 50) 

NMF + QR : **un fit par $p \in [1, 47]$**.

SVD : **un seul fit** (toutes les tranches $p$ sont des troncatures).

Les agrégats synthétiques sont formés à partir du panel public,

In [4]:
P_LIST = list(range(1, 48))                    

def _build_matched_bases():
    # agregats synthetiques de taille n_target (panel public) 
    X = basis.synthetic_aggregates(panel_tensor, cfg.n_target, cfg.n_groups,
                                   cfg.fit_subsample, seed=cfg.seed)

    # SVD
    t0 = time.perf_counter()
    Q_full = basis.fit_svd_basis(X, max(P_LIST))
    svd = {p: Q_full[:, :p] for p in P_LIST}
    t_svd = time.perf_counter() - t0

    # NMF
    t0 = time.perf_counter()
    nmf = {p: basis.fit_nmf_basis(X, p, cfg.nmf_max_iter, cfg.nmf_tol)
           for p in tqdm(P_LIST, desc='NMF (fit + QR par p)')}
    t_nmf = time.perf_counter() - t0

    return dict(svd=svd, nmf=nmf, X=X, t_svd=t_svd, t_nmf=t_nmf, p_list=P_LIST)

bases = compute_or_load(cfg.key('matched_basis', agg=cfg.n_target), _build_matched_bases)
Q_svd, Q_nmf = bases['svd'], bases['nmf']

print(f"X (agregats synthetiques) : {bases['X'].shape}")
print(f"SVD : 1 fit  ->  {len(P_LIST)} tranches p   en {bases['t_svd']:.2f} s")
print(f"NMF : {len(P_LIST)} fits (+QR)             en {bases['t_nmf']:.2f} s"
      f"   ({1000*bases['t_nmf']/len(P_LIST):.0f} ms / fit)")

[cache] 'matched_basis__N1000_np500_nt50_s0_agg50' rechargé (joblib).
X (agregats synthetiques) : (10000, 48)
SVD : 1 fit  ->  47 tranches p   en 0.10 s
NMF : 47 fits (+QR)             en 146.99 s   (3128 ms / fit)


## Visualisation des bases d'ondelettes

In [5]:
def _sign_fix(W):
    s = np.sign(W[np.argmax(np.abs(W), axis=0), np.arange(W.shape[1])])
    return W * s

@interact(p=IntSlider(min=1, max=max(Q_svd), step=1, value=4, description='p',
                      continuous_update=False, layout=dict(width='430px')))
def show_bases(p):
    slots = np.arange(48)
    pts  = [0.5] if p == 1 else list(np.linspace(0, 1, p))
    cols = viz.sample_colorscale(viz.SCALE_SAISON, pts)       
    Wn, Ws = _sign_fix(Q_nmf[p]), _sign_fix(Q_svd[p])

    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.07,
                        subplot_titles=('NMF + QR', 'SVD'))
    for k in range(p):
        name = f"w_{k+1}"
        fig.add_trace(go.Scatter(x=slots, y=Wn[:, k], mode='lines', name=name,
                      legendgroup=name, line=dict(color=cols[k], width=1.6)), row=1, col=1)
        fig.add_trace(go.Scatter(x=slots, y=Ws[:, k], mode='lines', name=name,
                      legendgroup=name, showlegend=False,
                      line=dict(color=cols[k], width=1.6)), row=1, col=2)
    fig.update_xaxes(title_text='créneau')
    fig.update_yaxes(title_text='amplitude', row=1, col=1)
    fig.update_layout(
        height=430, width=1000, template='plotly_white',
        title=dict(text=f"Bases d'ondelettes (p = {p})", x=0.5,
                   font=dict(size=15, color=viz.COLOR_REAL)),
        legend=dict(title='vecteur', x=1.02, y=0.5, yanchor='middle',
                    bordercolor='#d5dbdb', borderwidth=1),
        margin=dict(t=60, b=50, r=110))
    fig.show()

interactive(children=(IntSlider(value=4, continuous_update=False, description='p', layout=Layout(width='430px'…

## Reconstruction du signal 

In [6]:
@lru_cache(maxsize=None)
def _proj(p):
    _, pn, en = basis.project_on_basis(df_agg, Q_nmf[p])     # repro + erreurs par jour (NMF)
    _, ps, es = basis.project_on_basis(df_agg, Q_svd[p])     # repro + erreurs par jour (SVD)
    return pn.values, en, ps.values, es

PROJ_METRICS = list(_proj(1)[1].columns)     
def _fmt(metric, v):
    return f"{v:.1f}%" if metric in ('MAPE', 'NMAE') else f"{v:.3f}"

T, slots = len(df_agg), np.arange(48)

@interact(metric=Dropdown(options=[(metrics.METRIC_LABEL[m], m) for m in PROJ_METRICS],
                          value='NMAE', description='metrique',
                          layout=dict(width='300px'), style={'description_width': '80px'}),
          p=IntSlider(min=1, max=max(Q_svd), step=1, value=8, description='p',
                      continuous_update=False, layout=dict(width='430px')),
          day=IntSlider(min=0, max=T - 1, step=1, value=0, description='jour',
                        continuous_update=False, layout=dict(width='430px')))
def show_reconstruction(metric, p, day):
    Ln, en, Ls, es = _proj(p)
    mn, ms = _fmt(metric, en[metric].iloc[day]), _fmt(metric, es[metric].iloc[day])

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=slots, y=df_agg.values[day], mode='lines', name='reel',
                  line=dict(color=viz.COLOR_REAL, width=2.6)))
    fig.add_trace(go.Scatter(x=slots, y=Ln[day], mode='lines', name=f"NMF  ({metric} = {mn})",
                  line=dict(color=viz.COLOR_NMF, width=1.8)))
    fig.add_trace(go.Scatter(x=slots, y=Ls[day], mode='lines', name=f"SVD  ({metric} = {ms})",
                  line=dict(color=viz.COLOR_SVD, width=1.8, dash='dot')))
    fig.update_layout(
        title=dict(text=f"Reconstruction de l'agregat cible  -  p = {p}  -  {df_agg.index[day]:%d %b %Y}",
                   x=0.5, font=dict(size=15, color=viz.COLOR_REAL)),
        xaxis_title='creneau', yaxis_title='charge (kVA)',
        height=460, width=900, template='plotly_white',
        legend=dict(x=0.01, y=0.99, xanchor='left', yanchor='top',
                    bgcolor='rgba(255,255,255,0.85)', bordercolor='#d5dbdb', borderwidth=1),
        margin=dict(t=60, b=55, l=65, r=20))
    fig.update_yaxes(rangemode='tozero')
    fig.show()

interactive(children=(Dropdown(description='metrique', index=1, layout=Layout(width='300px'), options=(('MAPE …

## Erreur de projection 

In [7]:
@interact(metric=Dropdown(options=[(metrics.METRIC_LABEL[m], m) for m in PROJ_METRICS],
                          value='NMAE', description='metrique',
                          layout=dict(width='300px'), style={'description_width': '80px'}),
          p=IntSlider(min=1, max=max(Q_svd), step=1, value=8, description='p',
                      continuous_update=False, layout=dict(width='430px')))
def show_metric_time_box(metric, p):
    _, en, _, es = _proj(p)
    yn, ys = en[metric].values, es[metric].values          # erreur par jour (NMF / SVD)
    fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                        column_widths=[0.74, 0.26], horizontal_spacing=0.04,
                        subplot_titles=('evolution temporelle', 'distribution'))
    fig.add_trace(go.Scatter(x=df_agg.index, y=yn, mode='lines', name='NMF',
                  line=dict(color=viz.COLOR_NMF, width=1.5)), row=1, col=1)
    fig.add_trace(go.Scatter(x=df_agg.index, y=ys, mode='lines', name='SVD',
                  line=dict(color=viz.COLOR_SVD, width=1.5, dash='dot')), row=1, col=1)
    fig.add_trace(go.Box(y=yn, name='NMF', marker_color=viz.COLOR_NMF,
                  boxpoints=False, showlegend=False), row=1, col=2)
    fig.add_trace(go.Box(y=ys, name='SVD', marker_color=viz.COLOR_SVD,
                  boxpoints=False, showlegend=False), row=1, col=2)
    fig.update_xaxes(tickformat='%b', dtick='M1', row=1, col=1)
    fig.update_yaxes(title_text=metrics.METRIC_LABEL[metric], row=1, col=1)
    fig.update_layout(
        title=dict(text=f"Erreur de projection par jour  -  p = {p}  -  {metrics.METRIC_LABEL[metric]}",
                   x=0.5, font=dict(size=15, color=viz.COLOR_REAL)),
        height=460, width=1050, template='plotly_white',
        margin=dict(t=100),
        legend=dict(orientation='h', yanchor='bottom', y=1.06, xanchor='center', x=0.5))
    fig.show()

interactive(children=(Dropdown(description='metrique', index=1, layout=Layout(width='300px'), options=(('MAPE …

## Coefficients d'ondelettes

### Trajectoires

In [8]:
Q = {'NMF': Q_nmf, 'SVD': Q_svd}

@lru_cache(maxsize=None)
def _coeffs(method, p):
    return df_agg.values @ Q[method][p]      # (T, p) 

@interact(method=Dropdown(options=['NMF', 'SVD'], value='NMF', description='base',
                          layout=dict(width='220px'), style={'description_width': '60px'}),
          p=IntSlider(min=1, max=max(Q_svd), step=1, value=4, description='p',
                      continuous_update=False, layout=dict(width='430px')))
def show_coeffs(method, p):
    A = _coeffs(method, p)                                 # (T, p)
    pts  = [0.5] if p == 1 else list(np.linspace(0, 1, p))
    cols = viz.sample_colorscale(viz.SCALE_SAISON, pts)    

    fig = go.Figure()
    for k in range(p):
        fig.add_trace(go.Scatter(x=df_agg.index, y=A[:, k], mode='lines',
                      name=f"a_{k+1}", line=dict(color=cols[k], width=1.3)))
    fig.update_layout(
        title=dict(text=f"Trajectoires des coefficients d'ondelette  -  {method}  -  p = {p}",
                   x=0.5, font=dict(size=15, color=viz.COLOR_REAL)),
        xaxis_title='jour', yaxis_title='coefficient',
        height=460, width=1000, template='plotly_white',
        legend=dict(title='coef', x=1.02, y=0.5, yanchor='middle',
                    bordercolor='#d5dbdb', borderwidth=1),
        margin=dict(t=60, b=50, r=110))
    fig.update_xaxes(tickformat='%b', dtick='M1')
    fig.show()

interactive(children=(Dropdown(description='base', layout=Layout(width='220px'), options=('NMF', 'SVD'), style…

### Distribution

In [9]:
@interact(method=Dropdown(options=['NMF', 'SVD'], value='SVD', description='base',
                          layout=dict(width='220px'), style={'description_width': '60px'}),
          p=IntSlider(min=1, max=max(Q_svd), step=1, value=8, description='p',
                      continuous_update=False, layout=dict(width='430px')))
def show_coeff_boxplots(method, p):
    A = _coeffs(method, p)                                  # (T, p) 
    pts  = [0.5] if p == 1 else list(np.linspace(0, 1, p))
    cols = viz.sample_colorscale(viz.SCALE_SAISON, pts)     

    fig = go.Figure()
    for k in range(p):
        fig.add_trace(go.Box(y=A[:, k], name=f"a_{k+1}", marker_color=cols[k],
                             boxpoints=False, line=dict(width=1.2), showlegend=False))
    fig.update_layout(
        title=dict(text=f"Distribution des coefficients d'ondelette  -  {method}  -  p = {p}",
                   x=0.5, font=dict(size=15, color=viz.COLOR_REAL)),
        xaxis_title='coefficient', yaxis_title='valeur',
        height=460, width=min(1300, 240 + 55 * p), template='plotly_white')
    fig.update_yaxes(zeroline=True, zerolinecolor='#bbb')
    fig.show()

interactive(children=(Dropdown(description='base', index=1, layout=Layout(width='220px'), options=('NMF', 'SVD…

### Corrélations

In [10]:
@interact(method=Dropdown(options=['NMF', 'SVD'], value='SVD', description='base',
                          layout=dict(width='220px'), style={'description_width': '60px'}),
          p=IntSlider(min=2, max=max(Q_svd), step=1, value=8, description='p',
                      continuous_update=False, layout=dict(width='430px')))
def show_coeff_corr(method, p):
    A  = _coeffs(method, p)                       # (T, p) : trajectoires (cache)
    dA = np.diff(A, axis=0)                        # (T-1, p) : increments journaliers
    C_traj = np.corrcoef(A,  rowvar=False)         # (p, p)
    C_incr = np.corrcoef(dA, rowvar=False)         # (p, p)
    idx = list(range(1, p + 1))

    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.13,
                        subplot_titles=('correlation des trajectoires',
                                        'correlation des increments journaliers'))
    for col, C, show in [(1, C_traj, False), (2, C_incr, True)]:
        fig.add_trace(go.Heatmap(z=C, x=idx, y=idx, zmin=-1, zmax=1, zmid=0,
                      colorscale=viz.SCALE_CORR, reversescale=False, showscale=show,
                      colorbar=dict(title='corr', thickness=14, len=0.85, x=1.02)),
                      row=1, col=col)
    fig.update_xaxes(title_text='coefficient')
    fig.update_yaxes(title_text='coefficient', autorange='reversed')  
    fig.update_layout(
        title=dict(text=f"corrélation des coefficients  -  {method}  -  p = {p}",
                   x=0.5, font=dict(size=15, color=viz.COLOR_REAL)),
        height=470, width=1000, template='plotly_white',
        margin=dict(t=80, b=55, r=90))
    fig.show()

interactive(children=(Dropdown(description='base', index=1, layout=Layout(width='220px'), options=('NMF', 'SVD…

## Compromis erreur projection / perturbation pour le choix de p

Pour chaque jour $t=1,\dots,T$ on a :

$$\underbrace{L(t)}_{\text{réel}},\qquad
\underbrace{\hat L(t)=W W^\top L(t)}_{\text{projeté}},\qquad
\underbrace{\tilde L(t)=W\big(\alpha(t)+b(t)\big)}_{\text{perturbé (publié)}}$$

Par inégalité triangulaire (valable pour les métriques adéquates, e.g. RMSE, MAE) :

$$\underbrace{\lVert L(t)- \tilde L(t)\rVert}_{\text{err. totale}}\ \le\
\underbrace{\lVert L(t)- \hat L(t)\rVert}_{\text{projection}}+
\underbrace{\lVert \hat L(t)-\tilde L(t)\rVert}_{\text{perturbation}}$$

L'erreur de projection décroit avec p ( meilleur reconstruction du signal ) tandis que le terme de perturbation croit avec p ( plus de coefficients à bruiter ) 
On souhaite choisir le rang p qui minimise l'erreur totale de notre mécanisme.


In [11]:
def _tradeoff(method, eps, mode='per_day'):
    Qdict = Q_nmf if method == 'NMF' else Q_svd
    P = np.array(sorted(Qdict.keys()))
    L_true = df_agg.values
    panel_profiles = D['panel_profiles']
    target_rows = df_daily.loc[list(D['target_users'])]
    T = L_true.shape[0]

    keys = ['proj', 'pert_lap', 'pert_gau', 'tot_lap', 'tot_gau']
    acc = {m: {k: [] for k in keys} for m in metrics.METRICS}

    for p in P:
        W = Qdict[int(p)]
        Delta, Delta_mean = basis.panel_sensitivity(panel_profiles, W, cfg.n_target, cfg.clip_quantile)
        C, Delta2         = basis.panel_sensitivity_l2(panel_profiles, W, cfg.n_target, cfg.clip_quantile)

        alpha_clip = mechanisms.clip_aggregate_coeffs(target_rows, W, Delta)

        # ── projection géométrique pure (sans clipping) ───────────────────────
        L_hat_true = (L_true @ W) @ W.T

        L_lap = mechanisms.wpa_laplace(alpha_clip, W, Delta_mean, float(eps), mode=mode)
        if mode == 'per_day':
            sigma, _ = mechanisms.sigma_one_release(float(eps), cfg.delta_dp, Delta2)
        else:
            sigma, _ = mechanisms.sigma_for_total(float(eps), cfg.delta_dp, Delta2, T)
        L_gau = mechanisms.wpa_gaussian(alpha_clip, W, Delta2, sigma)

        m_proj = metrics.per_day_metrics(L_true, L_hat_true)   # ← projection pure
        m_lap  = metrics.per_day_metrics(L_true, L_lap)
        m_gau  = metrics.per_day_metrics(L_true, L_gau)
        m_pl   = metrics.per_day_metrics(L_hat_true, L_lap)    # ← référence cohérente
        m_pg   = metrics.per_day_metrics(L_hat_true, L_gau)

        for m in metrics.METRICS:
            acc[m]['proj'].append(float(m_proj[m].mean()))
            acc[m]['tot_lap'].append(float(m_lap[m].mean()))
            acc[m]['tot_gau'].append(float(m_gau[m].mean()))
            acc[m]['pert_lap'].append(float(m_pl[m].mean()))
            acc[m]['pert_gau'].append(float(m_pg[m].mean()))

    acc = {m: {k: np.array(v) for k, v in d.items()} for m, d in acc.items()}
    return P, acc


@interact(method=Dropdown(options=['NMF', 'SVD'], value='SVD', description='base',
                          layout=dict(width='200px'), style={'description_width': '55px'}),
          metric=Dropdown(options=[(metrics.METRIC_LABEL[m], m) for m in metrics.METRICS],
                          value='NMAE', description='metrique',
                          layout=dict(width='300px'), style={'description_width': '75px'}),
          eps=IntSlider(min=1, max=100, step=5, value=10, description='eps',
                        continuous_update=False, layout=dict(width='430px')),
          pmax=IntSlider(min=2, max=max(Q_svd), step=1, value=20, description='pmax',
                         continuous_update=False, layout=dict(width='430px')))
def show_tradeoff(method, metric, eps, pmax):
    P, acc = _tradeoff(method, eps)
    d, mask = acc[metric], P <= pmax
    x = P[mask]
    p_lap = int(x[np.argmin(d['tot_lap'][mask])])

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=d['proj'][mask], mode='lines', name='projection',
                  line=dict(color=viz.COLOR_PROJ, width=2.5)))
    fig.add_trace(go.Scatter(x=x, y=d['pert_lap'][mask], mode='lines', name='perturbation Laplace',
                  line=dict(color=viz.COLOR_GAUSSIAN, width=1.5)))
    fig.add_trace(go.Scatter(x=x, y=d['pert_gau'][mask], mode='lines', name='perturbation Gaussien',
                  line=dict(color=viz.COLOR_GAUSSIAN, width=1.5, dash='dot')))
    fig.add_trace(go.Scatter(x=x, y=d['tot_lap'][mask], mode='lines', name='total Laplace',
                  line=dict(color=viz.COLOR_LAPLACE, width=2.8)))
    fig.add_trace(go.Scatter(x=x, y=d['tot_gau'][mask], mode='lines', name='total Gaussien',
                  line=dict(color=viz.COLOR_GAUSSIAN, width=2.8, dash='dot')))

    fig.add_vline(x=p_lap, line=dict(color=viz.COLOR_LAPLACE, width=1.4, dash='dot'),
                  annotation_text=f"p*={p_lap}", annotation_position='top left',
                  annotation_font=dict(color=viz.COLOR_LAPLACE, size=11))

    fig.update_layout(
        title=dict(text=f"Compromis projection / perturbation  -  {method}  -  eps={eps}  -  {metrics.METRIC_LABEL[metric]}",
                   x=0.5, font=dict(size=14, color=viz.COLOR_REAL)),
        xaxis_title='p', yaxis_title=metrics.METRIC_LABEL[metric],
        height=480, width=900, template='plotly_white', hovermode='x unified',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5))
    fig.update_yaxes(type='linear')
    fig.show()

interactive(children=(Dropdown(description='base', index=1, layout=Layout(width='200px'), options=('NMF', 'SVD…

Piste de recherche : Passage mécanisme gaussien isotrope $\rightarrow$ anisotrope 

Le `wpa_gaussian` actuel injecte un bruit **isotrope** sur les coefficients $\alpha(t)\in\mathbb{R}^p$ :
$$ n(t)\sim\mathcal{N}(0,\sigma^2 I_p),\qquad \sigma^2=\frac{\Delta_2^2}{2\rho}, $$
où $\Delta_2$ est la sensibilité $L_2$ (estimée sur le panel) et $\rho$ le budget zCDP. **Le même $\sigma$ est appliqué à toutes les coordonnées.**

Or pour la **SVD**, l'énergie est concentrée dans les premiers coefficients : $\Delta_2$ est dominée par $\Delta_1$, et ce $\sigma$ calibré sur la coordonnée dominante, est versé tel quel sur les $\alpha_2,\dots,\alpha_p$ quasi plats.

**Gaussien anisotrope** : un $\sigma_j$ propre à chaque coordonnée,
$$ n(t)\sim\mathcal{N}\!\big(0,\operatorname{diag}(\sigma_1^2,\dots,\sigma_p^2)\big),\qquad
\rho=\tfrac12\sum_{j}\frac{\Delta_j^2}{\sigma_j^2}. $$
En **minimisant l'erreur** $\mathbb{E}[\lVert n\rVert_2^2]= Tr(Cov(n)) = \sum_j\sigma_j^2$ à budget $\rho$ fixé (Lagrange), on obtient
$$ \boxed{\;\sigma_j^2=\Delta_j\,\frac{\sum_k\Delta_k}{2\rho}\;}\qquad(\sigma_j\propto\sqrt{\Delta_j}),
\qquad\text{erreur totale}=\frac{\big(\sum_j\Delta_j\big)^2}{2\rho}. $$

Par Cauchy–Schwarz, $\big(\sum_j\Delta_j\big)^2\le p\sum_j\Delta_j^2$ : l'anisotrope est **toujours $\le$** l'isotrope, avec un gain d'autant plus grand que l'énergie est **concentrée** (cas SVD). On ne paie alors quasiment que pour $\alpha_1$, presque rien pour les coordonnées d'ordre élevé.

## Test d'un Generalized Additive Model (GAM)  (baseline + résidus) : conditionnement température / calendrier / classe Power $\times$ ToU

In [12]:
gam_model = compute_or_load(
    cfg.key('gam', backend='pygam'),
    lambda: gam.GAMResidualModelPyGAM(df_temp, df_label).fit(panel_ids, panel_tensor, dates))
baseline_target = gam_model.baseline_agg(target_users, df_temp, dates)   # (T, 48)
residual_hh     = gam_model.residual(panel_tensor, panel_ids, dates)     # (n_panel, T, 48) 
res_target      = df_agg.values - baseline_target                        # residu de la cible
print('baseline cible  | moy', round(float(np.abs(baseline_target).mean()), 3),
      '| residu moy', round(float(np.abs(res_target).mean()), 3))

[cache] 'gam__N1000_np500_nt50_s0_backendpygam' rechargé (joblib).
baseline cible  | moy 0.759 | residu moy 0.107


In [13]:
# Base SVD matchee sur les RESIDUS d'agregats de taille n_target (un seul fit), cachee
W_res_full = compute_or_load(
    cfg.key('res_svd', agg=cfg.n_target),
    lambda: basis.fit_svd_basis(
        basis.synthetic_aggregates(residual_hh, cfg.n_target, cfg.n_groups,
                                   cfg.fit_subsample, seed=cfg.seed),
        max(Q_svd)))                                    # (48, pmax)

L_true = df_agg.values

@lru_cache(maxsize=None)
def _recon(p):
    Wr = W_res_full[:, :p]
    L_gam = baseline_target + (res_target @ Wr) @ Wr.T          # GAM + SVD residus
    Ws    = Q_svd[p]
    L_raw = (L_true @ Ws) @ Ws.T                                # SVD brute (matchee)
    return metrics.per_day_metrics(L_true, L_gam), metrics.per_day_metrics(L_true, L_raw)

@interact(metric=Dropdown(options=[(metrics.METRIC_LABEL[m], m) for m in metrics.METRICS],
                          value='NMAE', description='metrique',
                          layout=dict(width='300px'), style={'description_width': '80px'}),
          p=IntSlider(min=1, max=W_res_full.shape[1], step=1, value=8, description='p',
                      continuous_update=False, layout=dict(width='430px')))
def show_gam_residual(metric, p):
    eg, er = _recon(p)
    yg, yr = eg[metric], er[metric]                            # erreur par jour
    fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                        column_widths=[0.74, 0.26], horizontal_spacing=0.04,
                        subplot_titles=('evolution temporelle', 'distribution'))
    fig.add_trace(go.Scatter(x=df_agg.index, y=yg, mode='lines', name='GAM + SVD residus',
                  line=dict(color=viz.COLOR_PROJ, width=1.8)), row=1, col=1)
    fig.add_trace(go.Scatter(x=df_agg.index, y=yr, mode='lines', name='SVD brute',
                  line=dict(color=viz.COLOR_SVD, width=1.5, dash='dot')), row=1, col=1)
    fig.add_trace(go.Box(y=yg, name='GAM + SVD residus', marker_color=viz.COLOR_PROJ,
                  boxpoints=False, showlegend=False), row=1, col=2)
    fig.add_trace(go.Box(y=yr, name='SVD brute', marker_color=viz.COLOR_SVD,
                  boxpoints=False, showlegend=False), row=1, col=2)
    fig.update_xaxes(tickformat='%b', dtick='M1', row=1, col=1)
    fig.update_yaxes(title_text=metrics.METRIC_LABEL[metric], row=1, col=1)
    fig.update_layout(
        title=dict(text=f"Reconstruction GAM + SVD residus  -  p = {p}  -  {metrics.METRIC_LABEL[metric]}",
                   x=0.5, font=dict(size=15, color=viz.COLOR_REAL)),
        height=460, width=1050, template='plotly_white', margin=dict(t=100),
        legend=dict(orientation='h', yanchor='bottom', y=1.06, xanchor='center', x=0.5))
    fig.show()

[cache] 'res_svd__N1000_np500_nt50_s0_agg50' rechargé (npz).


interactive(children=(Dropdown(description='metrique', index=1, layout=Layout(width='300px'), options=(('MAPE …